In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [4]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [5]:
X = df.drop(columns=["Class"],axis = 1)
Y = df["Class"]


df["Class"].unique().shape # 7 Classes: 7 output neurons

(7,)

### Encoding

In [6]:
le = LabelEncoder()

Y = le.fit_transform(Y)

In [7]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
)

### Scaling


In [8]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Deep Learning

In [9]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

In [10]:
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

Y_train_tensor = torch.tensor(
    Y_train,
    dtype=torch.long # Loss function: Cross Entropy Loss (Expects target values in long datatype)
)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

Y_test_tensor = torch.tensor(
    Y_test,
    dtype=torch.long
)

In [11]:
train_dataset = TensorDataset(
    X_train_tensor,
    Y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    Y_test_tensor
)

In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
)

In [13]:
class Classifier(nn.Module):

    def __init__(self):
        super(Classifier,self).__init__()

        self.model = nn.Sequential(
            # 1st Hidden Layer

            nn.Linear(in_features=X.shape[1],out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64,out_features=64),
            nn.ReLU(),
            nn.Linear(64,7)
        )

    def forward(self,x):
        return self.model(x)

In [14]:
model = Classifier()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [15]:
epochs = 100
best_model_loss = float("inf")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for xb,yb in train_loader:

        optimizer.zero_grad()

        labels = model(xb)
        loss = criterion(labels,yb)

        loss.backward()
        optimizer.step()

        running_loss+=loss.item()
    train_loss = running_loss/len(train_loader)
    print(f"Epoch #{epoch+1}/{epochs} || Loss: {train_loss}")
    if(train_loss < best_model_loss):
        print(f"Found better model at epoch #{epoch+1}")
        best_model_loss = train_loss
        torch.save(model.state_dict(),"best_model.pt")

Epoch #1/100 || Loss: 1.7125041329342385
Found better model at epoch #1
Epoch #2/100 || Loss: 1.1007549788640894
Found better model at epoch #2
Epoch #3/100 || Loss: 0.7392096143701802
Found better model at epoch #3
Epoch #4/100 || Loss: 0.5633626204469929
Found better model at epoch #4
Epoch #5/100 || Loss: 0.4562362860078397
Found better model at epoch #5
Epoch #6/100 || Loss: 0.39707201006619824
Found better model at epoch #6
Epoch #7/100 || Loss: 0.34750322025755176
Found better model at epoch #7
Epoch #8/100 || Loss: 0.3305043020974035
Found better model at epoch #8
Epoch #9/100 || Loss: 0.3020561240289522
Found better model at epoch #9
Epoch #10/100 || Loss: 0.2837282717227936
Found better model at epoch #10
Epoch #11/100 || Loss: 0.26071249823207443
Found better model at epoch #11
Epoch #12/100 || Loss: 0.2579345654534257
Found better model at epoch #12
Epoch #13/100 || Loss: 0.23903158436650815
Found better model at epoch #13
Epoch #14/100 || Loss: 0.22553815109574277
Found bet

In [16]:
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

# Evaluation 

In [31]:
model.eval()

total =0
correct = 0
all_predictions = []
all_actuals = []

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb) # [0.2,0.5,1.3,-0.5, ... ] 7 such values, where the value is maximum, that index will be predicted as the class
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == yb).sum().item()
        total+=yb.size(0)
        all_predictions.extend(predicted.numpy())
        all_actuals.extend(yb.numpy())

# Create comprehensive dataframe
classes = df["Class"]

comparison_df = pd.DataFrame({
    'Predicted': le.inverse_transform(all_predictions),
    'Actual': le.inverse_transform(all_actuals)
})
print(f"Total: {total} || Correct : {correct}")

Total: 180 || Correct : 172


In [33]:
comparison_df

,Predicted,Actual
0,DOKOL,DOKOL
1,SAFAVI,SAFAVI
2,DOKOL,DOKOL
3,SOGAY,SOGAY
4,BERHI,BERHI
...,...,...
175,SAFAVI,SAFAVI
176,BERHI,BERHI
177,DEGLET,DEGLET
178,BERHI,BERHI
